In [58]:
import requests
import pandas as pd
import time

# =========================
# 1. ZONES (Maroc découpé)
# =========================
zones = {
    "north": (34.5, -6.5, 36.0, -1.5),
    "center": (32.0, -8.5, 34.5, -6.0),
    "south": (27.5, -12.0, 32.0, -8.0),
}

# =========================
# 2. CONFIG
# =========================
url = "https://overpass-api.de/api/interpreter"

headers = {
    "User-Agent": "Mozilla/5.0"
}

all_restaurants = []

# =========================
# 3. LOOP ZONES
# =========================
for zone_name, bbox in zones.items():
    print(f"\n🔎 Zone: {zone_name}")

    query = f"""
    [out:json][timeout:180];
    (
      node["amenity"="restaurant"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
      way["amenity"="restaurant"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
      relation["amenity"="restaurant"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
    );
    out center;
    """

    data = None

    # =========================
    # 4. RETRY SYSTEM
    # =========================
    for i in range(3):
        try:
            r = requests.post(
                url,
                data={"data": query},
                headers=headers,
                timeout=240
            )

            print("Status:", r.status_code)

            if r.status_code != 200:
                print("HTTP error, retry...")
                time.sleep(10)
                continue

            if "json" not in r.headers.get("Content-Type", ""):
                print("Non JSON response, retry...")
                time.sleep(10)
                continue

            data = r.json()
            break

        except Exception as e:
            print("Retry error:", e)
            time.sleep(10)

    # =========================
    # 5. CHECK DATA
    # =========================
    if not data or not data.get("elements"):
        print(f"❌ No data for {zone_name}")
        continue

    # =========================
    # 6. PARSE DATA
    # =========================
    for el in data["elements"]:
        tags = el.get("tags", {})

        name = tags.get("name")
        if not name:
            continue

        lat = el.get("lat", el.get("center", {}).get("lat"))
        lon = el.get("lon", el.get("center", {}).get("lon"))

        all_restaurants.append({
            "name": name,
            "zone": zone_name,
            "lat": lat,
            "lon": lon,
            "cuisine": tags.get("cuisine"),
            "brand": tags.get("brand"),
            "phone": tags.get("phone"),
            "website": tags.get("website"),
            "amenity": tags.get("amenity")
        })

    print(f"✅ Zone {zone_name} done: {len(all_restaurants)} total so far")

    # pause important (évite 429/504)
    time.sleep(15)

# =========================
# 7. DATAFRAME FINAL
# =========================
df = pd.DataFrame(all_restaurants)

# clean
df = df.dropna(subset=["name"]).drop_duplicates()

print("\n📊 FINAL DATASET")
print(df.shape)
print(df.head())

# =========================
# 8. EXPORT CSV
# =========================
df.to_csv("morocco_restaurants.csv", index=False, encoding="utf-8-sig")

print("\n✅ CSV saved: morocco_restaurants.csv")


🔎 Zone: north
Status: 200
✅ Zone north done: 529 total so far

🔎 Zone: center
Status: 504
HTTP error, retry...
Status: 200
✅ Zone center done: 1284 total so far

🔎 Zone: south
Status: 429
HTTP error, retry...
Status: 200
✅ Zone south done: 1882 total so far

📊 FINAL DATASET
(1882, 9)
             name   zone        lat       lon cuisine brand phone website  \
0  Bodegas Madrid  north  35.295274 -2.941473    None  None  None    None   
1     Bar Sevilla  north  35.293182 -2.935821    None  None  None    None   
2          Sunset  north  35.759920 -5.938810    None  None  None    None   
3  Marina Hercule  north  35.760242 -5.938881    None  None  None    None   
4        La Farma  north  35.665505 -5.307170    None  None  None    None   

      amenity  
0  restaurant  
1  restaurant  
2  restaurant  
3  restaurant  
4  restaurant  

✅ CSV saved: morocco_restaurants.csv


In [59]:
import requests
import pandas as pd
import time

# =========================
# 1. VILLES MAROC + BBOX
# =========================
cities = {
    "Casablanca": (33.48, -7.75, 33.75, -7.45),
    "Rabat": (33.90, -6.90, 34.05, -6.75),
    "Marrakech": (31.55, -8.10, 31.75, -7.85),
    "Fès": (34.00, -5.10, 34.10, -4.90),
    "Tanger": (35.70, -5.95, 35.85, -5.70),
    "Agadir": (30.35, -9.60, 30.50, -9.40),
    "Meknès": (33.85, -5.60, 34.00, -5.40),
    "Oujda": (34.65, -1.95, 34.80, -1.80),
    "Kenitra": (34.22, -6.70, 34.32, -6.55),
    "Tétouan": (35.55, -5.45, 35.65, -5.30),
    "Safi": (32.25, -9.25, 32.35, -9.10),
    "El Jadida": (33.20, -8.55, 33.30, -8.40),
    "Nador": (35.10, -2.95, 35.20, -2.80),
    "Beni Mellal": (32.30, -6.40, 32.40, -6.25),
    "Khouribga": (32.85, -6.95, 32.95, -6.80),
    "Essaouira": (31.50, -9.80, 31.60, -9.65),
    "Ouarzazate": (30.90, -6.95, 31.05, -6.80),
    "Berkane": (34.90, -2.40, 35.00, -2.25),
    "Larache": (35.15, -6.15, 35.25, -6.00),
    "Settat": (33.00, -7.65, 33.10, -7.50)
}

# =========================
# 2. CONFIG
# =========================
url = "https://overpass-api.de/api/interpreter"

headers = {
    "User-Agent": "Mozilla/5.0"
}

all_data = []

# =========================
# 3. LOOP VILLES
# =========================
for city, bbox in cities.items():
    print(f"\n🔎 Scraping {city} ...")

    query = f"""
    [out:json][timeout:180];
    (
      node["amenity"="restaurant"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
      way["amenity"="restaurant"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
      relation["amenity"="restaurant"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
    );
    out center;
    """

    data = None

    # =========================
    # 4. RETRY SYSTEM (405 / 429 / 504)
    # =========================
    for i in range(3):
        try:
            r = requests.post(
                url,
                data={"data": query},
                headers=headers,
                timeout=240
            )

            print(f"{city} Status:", r.status_code)

            # 🔥 retry sur erreurs HTTP
            if r.status_code in [429, 504, 405]:
                print(f"⚠️ {city} retry ({r.status_code})")
                time.sleep(10 * (i + 1))
                continue

            if r.status_code != 200:
                print(f"❌ {city} error")
                time.sleep(10)
                continue

            if "json" not in r.headers.get("Content-Type", ""):
                print(f"❌ {city} non JSON")
                time.sleep(10)
                continue

            data = r.json()
            break

        except Exception as e:
            print(f"Retry error {city}:", e)
            time.sleep(10)

    # =========================
    # 5. CHECK DATA
    # =========================
    if not data or not data.get("elements"):
        print(f"❌ No data {city}")
        continue

    # =========================
    # 6. PARSING
    # =========================
    for el in data["elements"]:
        tags = el.get("tags", {})

        name = tags.get("name")
        if not name:
            continue

        lat = el.get("lat", el.get("center", {}).get("lat"))
        lon = el.get("lon", el.get("center", {}).get("lon"))

        all_data.append({
            "name": name,
            "city": city,
            "lat": lat,
            "lon": lon,
            "cuisine": tags.get("cuisine"),
            "brand": tags.get("brand"),
            "phone": tags.get("phone"),
            "website": tags.get("website"),
            "amenity": tags.get("amenity")
        })

    print(f"✅ {city} done | total: {len(all_data)}")

    # 🔥 IMPORTANT anti-bloquage
    time.sleep(12)

# =========================
# 7. DATAFRAME FINAL
# =========================
df = pd.DataFrame(all_data)

df = df.dropna(subset=["name"]).drop_duplicates()

print("\n📊 FINAL RESULT")
print(df.shape)
print(df.head())

# =========================
# 8. EXPORT CSV
# =========================
df.to_csv("morocco_restaurants_cities.csv", index=False, encoding="utf-8-sig")

print("\n✅ CSV saved!")


🔎 Scraping Casablanca ...
Casablanca Status: 200
✅ Casablanca done | total: 327

🔎 Scraping Rabat ...
Rabat Status: 504
⚠️ Rabat retry (504)
Rabat Status: 504
⚠️ Rabat retry (504)
Rabat Status: 504
⚠️ Rabat retry (504)
❌ No data Rabat

🔎 Scraping Marrakech ...
Marrakech Status: 200
✅ Marrakech done | total: 671

🔎 Scraping Fès ...
Fès Status: 504
⚠️ Fès retry (504)
Fès Status: 200
✅ Fès done | total: 991

🔎 Scraping Tanger ...
Tanger Status: 429
⚠️ Tanger retry (429)
Tanger Status: 200
✅ Tanger done | total: 1174

🔎 Scraping Agadir ...
Agadir Status: 504
⚠️ Agadir retry (504)
Agadir Status: 200
✅ Agadir done | total: 1235

🔎 Scraping Meknès ...
Meknès Status: 504
⚠️ Meknès retry (504)
Meknès Status: 200
✅ Meknès done | total: 1289

🔎 Scraping Oujda ...
Oujda Status: 504
⚠️ Oujda retry (504)
Oujda Status: 504
⚠️ Oujda retry (504)
Oujda Status: 504
⚠️ Oujda retry (504)
❌ No data Oujda

🔎 Scraping Kenitra ...
Kenitra Status: 504
⚠️ Kenitra retry (504)
Kenitra Status: 200
✅ Kenitra done |

In [64]:
import requests
import pandas as pd
import time

# =========================
# 1. VILLES MAROC + BBOX
# =========================
cities = {
    "Casablanca": (33.48, -7.75, 33.75, -7.45),
    "Rabat": (33.90, -6.90, 34.05, -6.75),
    "Marrakech": (31.55, -8.10, 31.75, -7.85),
    "Fès": (34.00, -5.10, 34.10, -4.90),
    "Tanger": (35.70, -5.95, 35.85, -5.70),
    "Agadir": (30.35, -9.60, 30.50, -9.40),
    "Meknès": (33.85, -5.60, 34.00, -5.40),
    "Oujda": (34.65, -1.95, 34.80, -1.80),
    "Kenitra": (34.22, -6.70, 34.32, -6.55),
    "Tétouan": (35.55, -5.45, 35.65, -5.30),
    "Safi": (32.25, -9.25, 32.35, -9.10),
    "El Jadida": (33.20, -8.55, 33.30, -8.40),
    "Nador": (35.10, -2.95, 35.20, -2.80),
    "Beni Mellal": (32.30, -6.40, 32.40, -6.25),
    "Khouribga": (32.85, -6.95, 32.95, -6.80),
    "Essaouira": (31.50, -9.80, 31.60, -9.65),
    "Ouarzazate": (30.90, -6.95, 31.05, -6.80),
    "Berkane": (34.90, -2.40, 35.00, -2.25),
    "Larache": (35.15, -6.15, 35.25, -6.00),
    "Settat": (33.00, -7.65, 33.10, -7.50)
}

# =========================
# 2. CONFIG
# =========================
url = "https://overpass-api.de/api/interpreter"
headers = {"User-Agent": "Mozilla/5.0"}

all_data = []

# =========================
# 3. LOOP VILLES
# =========================
for city, bbox in cities.items():
    print(f"\n☕ Scraping cafés {city} ...")

    query = f"""
    [out:json][timeout:180];
    (
      node["amenity"="cafe"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
      way["amenity"="cafe"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
      relation["amenity"="cafe"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
    );
    out center;
    """

    data = None

    # =========================
    # 4. RETRY SYSTEM
    # =========================
    for i in range(3):
        try:
            r = requests.post(
                url,
                data={"data": query},
                headers=headers,
                timeout=240
            )

            print(f"{city} Status:", r.status_code)

            if r.status_code in [429, 504, 405]:
                print(f"⚠️ retry {city} ({r.status_code})")
                time.sleep(10 * (i + 1))
                continue

            if r.status_code != 200:
                print(f"❌ error {city}")
                time.sleep(10)
                continue

            if "json" not in r.headers.get("Content-Type", ""):
                print(f"❌ non JSON {city}")
                time.sleep(10)
                continue

            data = r.json()
            break

        except Exception as e:
            print(f"Retry error {city}:", e)
            time.sleep(10)

    # =========================
    # 5. CHECK DATA
    # =========================
    if not data or not data.get("elements"):
        print(f"❌ No data {city}")
        continue

    # =========================
    # 6. PARSING
    # =========================
    for el in data["elements"]:
        tags = el.get("tags", {})

        name = tags.get("name")
        if not name:
            continue

        lat = el.get("lat", el.get("center", {}).get("lat"))
        lon = el.get("lon", el.get("center", {}).get("lon"))

        all_data.append({
            "name": name,
            "city": city,
            "lat": lat,
            "lon": lon,
            "cuisine": tags.get("cuisine"),
            "brand": tags.get("brand"),
            "phone": tags.get("phone"),
            "website": tags.get("website"),
            "amenity": "cafe"
        })

    print(f"✅ {city} done | total cafés: {len(all_data)}")

    time.sleep(12)

# =========================
# 7. DATAFRAME FINAL
# =========================
df = pd.DataFrame(all_data)

df = df.dropna(subset=["name"]).drop_duplicates()

print("\n📊 FINAL CAFÉS DATASET")
print(df.shape)
print(df.head())

# =========================
# 8. EXPORT CSV
# =========================
df.to_csv("morocco_cafes_cities.csv", index=False, encoding="utf-8-sig")

print("\n✅ CSV saved: morocco_cafes_cities.csv")


☕ Scraping cafés Casablanca ...
Casablanca Status: 200
✅ Casablanca done | total cafés: 516

☕ Scraping cafés Rabat ...
Rabat Status: 504
⚠️ retry Rabat (504)
Rabat Status: 504
⚠️ retry Rabat (504)
Rabat Status: 504
⚠️ retry Rabat (504)
❌ No data Rabat

☕ Scraping cafés Marrakech ...
Marrakech Status: 200
✅ Marrakech done | total cafés: 684

☕ Scraping cafés Fès ...
Fès Status: 504
⚠️ retry Fès (504)
Fès Status: 504
⚠️ retry Fès (504)
Fès Status: 504
⚠️ retry Fès (504)
❌ No data Fès

☕ Scraping cafés Tanger ...
Tanger Status: 200
✅ Tanger done | total cafés: 808

☕ Scraping cafés Agadir ...
Agadir Status: 504
⚠️ retry Agadir (504)
Agadir Status: 200
✅ Agadir done | total cafés: 851

☕ Scraping cafés Meknès ...
Meknès Status: 504
⚠️ retry Meknès (504)
Meknès Status: 504
⚠️ retry Meknès (504)
Meknès Status: 200
✅ Meknès done | total cafés: 1013

☕ Scraping cafés Oujda ...
Oujda Status: 504
⚠️ retry Oujda (504)
Oujda Status: 200
✅ Oujda done | total cafés: 1158

☕ Scraping cafés Kenitra 

In [71]:
import requests
import pandas as pd
import time

# =========================
# 1. VILLES MAROC + BBOX
# =========================
cities = {
    "Casablanca": (33.48, -7.75, 33.75, -7.45),
    "Rabat": (33.90, -6.90, 34.05, -6.75),
    "Marrakech": (31.55, -8.10, 31.75, -7.85),
    "Fès": (34.00, -5.10, 34.10, -4.90),
    "Tanger": (35.70, -5.95, 35.85, -5.70),
    "Agadir": (30.35, -9.60, 30.50, -9.40),
    "Meknès": (33.85, -5.60, 34.00, -5.40),
    "Oujda": (34.65, -1.95, 34.80, -1.80),
    "Kenitra": (34.22, -6.70, 34.32, -6.55),
    "Tétouan": (35.55, -5.45, 35.65, -5.30),
    "Safi": (32.25, -9.25, 32.35, -9.10),
    "El Jadida": (33.20, -8.55, 33.30, -8.40),
    "Nador": (35.10, -2.95, 35.20, -2.80),
    "Beni Mellal": (32.30, -6.40, 32.40, -6.25),
    "Essaouira": (31.50, -9.80, 31.60, -9.65),
    "Ouarzazate": (30.90, -6.95, 31.05, -6.80),
    "Berkane": (34.90, -2.40, 35.00, -2.25),
    "Larache": (35.15, -6.15, 35.25, -6.00),
    "Settat": (33.00, -7.65, 33.10, -7.50)
}

# =========================
# 2. CONFIG
# =========================
url = "https://overpass-api.de/api/interpreter"
headers = {"User-Agent": "Mozilla/5.0"}

all_data = []

# =========================
# 3. LOOP VILLES
# =========================
for city, bbox in cities.items():
    print(f"\n🏧 Scraping ATM {city} ...")

    query = f"""
    [out:json][timeout:180];
    (
      node["amenity"="atm"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
      way["amenity"="atm"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});

      node["man_made"="atm"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
      way["man_made"="atm"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});

      node["amenity"="bank"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
      way["amenity"="bank"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
    );
    out center;
    """

    data = None

    # =========================
    # 4. RETRY SYSTEM ROBUSTE
    # =========================
    for i in range(3):
        try:
            r = requests.post(
                url,
                data={"data": query},
                headers=headers,
                timeout=240
            )

            print(f"{city} Status:", r.status_code)

            # retry sur erreurs serveur
            if r.status_code in [429, 504, 405]:
                print(f"⚠️ retry {city} ({r.status_code})")
                time.sleep(15 * (i + 1))
                continue

            if r.status_code != 200:
                print(f"❌ error {city}")
                time.sleep(10)
                continue

            if "json" not in r.headers.get("Content-Type", ""):
                print(f"❌ non JSON {city}")
                time.sleep(10)
                continue

            data = r.json()
            break

        except Exception as e:
            print(f"Retry error {city}:", e)
            time.sleep(10)

    # =========================
    # 5. CHECK DATA
    # =========================
    if not data or not data.get("elements"):
        print(f"❌ No data {city}")
        continue

    # =========================
    # 6. PARSING INTELLIGENT
    # =========================
    for el in data["elements"]:
        tags = el.get("tags", {})

        lat = el.get("lat", el.get("center", {}).get("lat"))
        lon = el.get("lon", el.get("center", {}).get("lon"))

        if not lat or not lon:
            continue

        all_data.append({
            "name": tags.get("name", "ATM"),
            "city": city,
            "lat": lat,
            "lon": lon,
            "bank": tags.get("operator") or tags.get("brand"),
            "type": tags.get("amenity") or tags.get("man_made"),
            "level": "atm/bank"
        })

    print(f"✅ {city} done | total ATM+bank: {len(all_data)}")

    time.sleep(12)

# =========================
# 7. DATAFRAME FINAL
# =========================
df = pd.DataFrame(all_data)

df = df.dropna(subset=["lat", "lon"]).drop_duplicates()

print("\n📊 FINAL ATM + BANK DATASET")
print(df.shape)
print(df.head())

# =========================
# 8. EXPORT CSV
# =========================
df.to_csv("morocco_atm_banks_improved.csv", index=False, encoding="utf-8-sig")

print("\n✅ CSV saved: morocco_atm_banks_improved.csv")


🏧 Scraping ATM Casablanca ...
Casablanca Status: 504
⚠️ retry Casablanca (504)
Casablanca Status: 200
✅ Casablanca done | total ATM+bank: 819

🏧 Scraping ATM Rabat ...
Rabat Status: 504
⚠️ retry Rabat (504)
Rabat Status: 200
✅ Rabat done | total ATM+bank: 1067

🏧 Scraping ATM Marrakech ...
Marrakech Status: 504
⚠️ retry Marrakech (504)
Marrakech Status: 504
⚠️ retry Marrakech (504)
Marrakech Status: 504
⚠️ retry Marrakech (504)
❌ No data Marrakech

🏧 Scraping ATM Fès ...
Fès Status: 200
✅ Fès done | total ATM+bank: 1316

🏧 Scraping ATM Tanger ...
Tanger Status: 200
✅ Tanger done | total ATM+bank: 1446

🏧 Scraping ATM Agadir ...
Agadir Status: 429
⚠️ retry Agadir (429)
Agadir Status: 504
⚠️ retry Agadir (504)
Agadir Status: 200
✅ Agadir done | total ATM+bank: 1522

🏧 Scraping ATM Meknès ...
Meknès Status: 504
⚠️ retry Meknès (504)
Meknès Status: 200
✅ Meknès done | total ATM+bank: 1658

🏧 Scraping ATM Oujda ...
Oujda Status: 504
⚠️ retry Oujda (504)
Oujda Status: 504
⚠️ retry Oujda (50

In [75]:
import requests
import pandas as pd
import time

# =========================
# 1. VILLES MAROC + BBOX
# =========================
cities = {
    "Casablanca": (33.48, -7.75, 33.75, -7.45),
    "Rabat": (33.90, -6.90, 34.05, -6.75),
    "Marrakech": (31.55, -8.10, 31.75, -7.85),
    "Fès": (34.00, -5.10, 34.10, -4.90),
    "Tanger": (35.70, -5.95, 35.85, -5.70),
    "Agadir": (30.35, -9.60, 30.50, -9.40),
    "Meknès": (33.85, -5.60, 34.00, -5.40),
    "Oujda": (34.65, -1.95, 34.80, -1.80),
    "Kenitra": (34.22, -6.70, 34.32, -6.55),
    "Tétouan": (35.55, -5.45, 35.65, -5.30),
    "Safi": (32.25, -9.25, 32.35, -9.10),
    "El Jadida": (33.20, -8.55, 33.30, -8.40),
    "Nador": (35.10, -2.95, 35.20, -2.80),
    "Beni Mellal": (32.30, -6.40, 32.40, -6.25),
    "Essaouira": (31.50, -9.80, 31.60, -9.65),
    "Ouarzazate": (30.90, -6.95, 31.05, -6.80),
    "Berkane": (34.90, -2.40, 35.00, -2.25),
    "Larache": (35.15, -6.15, 35.25, -6.00),
    "Settat": (33.00, -7.65, 33.10, -7.50)
}

# =========================
# 2. CONFIG
# =========================
url = "https://overpass-api.de/api/interpreter"
headers = {"User-Agent": "Mozilla/5.0"}

all_data = []

# =========================
# 3. LOOP VILLES
# =========================
for city, bbox in cities.items():
    print(f"\n💊 Scraping pharmacies {city} ...")

    query = f"""
    [out:json][timeout:180];
    (
      node["amenity"="pharmacy"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
      way["amenity"="pharmacy"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
      relation["amenity"="pharmacy"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
    );
    out center;
    """

    data = None

    # =========================
    # 4. RETRY SYSTEM ROBUSTE
    # =========================
    for i in range(3):
        try:
            r = requests.post(
                url,
                data={"data": query},
                headers=headers,
                timeout=240
            )

            print(f"{city} Status:", r.status_code)

            if r.status_code in [429, 504, 405]:
                print(f"⚠️ retry {city} ({r.status_code})")
                time.sleep(15 * (i + 1))
                continue

            if r.status_code != 200:
                print(f"❌ error {city}")
                time.sleep(10)
                continue

            if "json" not in r.headers.get("Content-Type", ""):
                print(f"❌ non JSON {city}")
                time.sleep(10)
                continue

            data = r.json()
            break

        except Exception as e:
            print(f"Retry error {city}:", e)
            time.sleep(10)

    # =========================
    # 5. CHECK DATA
    # =========================
    if not data or not data.get("elements"):
        print(f"❌ No data {city}")
        continue

    # =========================
    # 6. PARSING
    # =========================
    for el in data["elements"]:
        tags = el.get("tags", {})

        name = tags.get("name")
        if not name:
            continue

        lat = el.get("lat", el.get("center", {}).get("lat"))
        lon = el.get("lon", el.get("center", {}).get("lon"))

        if not lat or not lon:
            continue

        all_data.append({
            "name": name,
            "city": city,
            "lat": lat,
            "lon": lon,
            "phone": tags.get("phone"),
            "website": tags.get("website"),
            "brand": tags.get("brand"),
            "type": "pharmacy"
        })

    print(f"✅ {city} done | total pharmacies: {len(all_data)}")

    time.sleep(12)

# =========================
# 7. DATAFRAME FINAL
# =========================
df = pd.DataFrame(all_data)

df = df.dropna(subset=["name"]).drop_duplicates()

print("\n📊 FINAL PHARMACY DATASET")
print(df.shape)
print(df.head())

# =========================
# 8. EXPORT CSV
# =========================
df.to_csv("morocco_pharmacies.csv", index=False, encoding="utf-8-sig")

print("\n✅ CSV saved: morocco_pharmacies.csv")


💊 Scraping pharmacies Casablanca ...
Casablanca Status: 504
⚠️ retry Casablanca (504)
Casablanca Status: 200
✅ Casablanca done | total pharmacies: 412

💊 Scraping pharmacies Rabat ...
Rabat Status: 504
⚠️ retry Rabat (504)
Rabat Status: 200
✅ Rabat done | total pharmacies: 651

💊 Scraping pharmacies Marrakech ...
Marrakech Status: 429
⚠️ retry Marrakech (429)
Marrakech Status: 200
✅ Marrakech done | total pharmacies: 1120

💊 Scraping pharmacies Fès ...
Fès Status: 200
✅ Fès done | total pharmacies: 1483

💊 Scraping pharmacies Tanger ...
Tanger Status: 429
⚠️ retry Tanger (429)
Tanger Status: 200
✅ Tanger done | total pharmacies: 1846

💊 Scraping pharmacies Agadir ...
Agadir Status: 200
✅ Agadir done | total pharmacies: 2150

💊 Scraping pharmacies Meknès ...
Meknès Status: 429
⚠️ retry Meknès (429)
Meknès Status: 504
⚠️ retry Meknès (504)
Meknès Status: 200
✅ Meknès done | total pharmacies: 2437

💊 Scraping pharmacies Oujda ...
Oujda Status: 504
⚠️ retry Oujda (504)
Oujda Status: 200
✅

In [76]:
import requests
import pandas as pd
import time

# =========================
# 1. VILLES MAROC + BBOX
# =========================
cities = {
    "Beni Mellal": (32.30, -6.40, 32.40, -6.25),
    "Larache": (35.15, -6.15, 35.25, -6.00),
    "Settat": (33.00, -7.65, 33.10, -7.50)
}

# =========================
# 2. CONFIG
# =========================
url = "https://overpass-api.de/api/interpreter"
headers = {"User-Agent": "Mozilla/5.0"}

all_data = []

# =========================
# 3. LOOP VILLES
# =========================
for city, bbox in cities.items():
    print(f"\n💊 Scraping pharmacies {city} ...")

    query = f"""
    [out:json][timeout:180];
    (
      node["amenity"="pharmacy"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
      way["amenity"="pharmacy"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
      relation["amenity"="pharmacy"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
    );
    out center;
    """

    data = None

    # =========================
    # 4. RETRY SYSTEM ROBUSTE
    # =========================
    for i in range(3):
        try:
            r = requests.post(
                url,
                data={"data": query},
                headers=headers,
                timeout=240
            )

            print(f"{city} Status:", r.status_code)

            if r.status_code in [429, 504, 405]:
                print(f"⚠️ retry {city} ({r.status_code})")
                time.sleep(15 * (i + 1))
                continue

            if r.status_code != 200:
                print(f"❌ error {city}")
                time.sleep(10)
                continue

            if "json" not in r.headers.get("Content-Type", ""):
                print(f"❌ non JSON {city}")
                time.sleep(10)
                continue

            data = r.json()
            break

        except Exception as e:
            print(f"Retry error {city}:", e)
            time.sleep(10)

    # =========================
    # 5. CHECK DATA
    # =========================
    if not data or not data.get("elements"):
        print(f"❌ No data {city}")
        continue

    # =========================
    # 6. PARSING
    # =========================
    for el in data["elements"]:
        tags = el.get("tags", {})

        name = tags.get("name")
        if not name:
            continue

        lat = el.get("lat", el.get("center", {}).get("lat"))
        lon = el.get("lon", el.get("center", {}).get("lon"))

        if not lat or not lon:
            continue

        all_data.append({
            "name": name,
            "city": city,
            "lat": lat,
            "lon": lon,
            "phone": tags.get("phone"),
            "website": tags.get("website"),
            "brand": tags.get("brand"),
            "type": "pharmacy"
        })

    print(f"✅ {city} done | total pharmacies: {len(all_data)}")

    time.sleep(12)

# =========================
# 7. DATAFRAME FINAL
# =========================
df = pd.DataFrame(all_data)

df = df.dropna(subset=["name"]).drop_duplicates()

print("\n📊 FINAL PHARMACY DATASET")
print(df.shape)
print(df.head())

# =========================
# 8. EXPORT CSV
# =========================
df.to_csv("morocco_pharmacies11.csv", index=False, encoding="utf-8-sig")

print("\n✅ CSV saved: morocco_pharmacies.csv")


💊 Scraping pharmacies Beni Mellal ...
Beni Mellal Status: 200
✅ Beni Mellal done | total pharmacies: 79

💊 Scraping pharmacies Larache ...
Larache Status: 504
⚠️ retry Larache (504)
Larache Status: 504
⚠️ retry Larache (504)
Larache Status: 200
✅ Larache done | total pharmacies: 98

💊 Scraping pharmacies Settat ...
Settat Status: 200
✅ Settat done | total pharmacies: 150

📊 FINAL PHARMACY DATASET
(150, 8)
                                                name         city        lat  \
0               Pharmacie El Hansali صيدلية الحنصالي  Beni Mellal  32.336915   
1  Pharmacie Quartier Administratif صيدلية الحي ا...  Beni Mellal  32.330090   
2                       Pharmacie Bourra صيدلية بورا  Beni Mellal  32.331873   
3                                             MyPARA  Beni Mellal  32.333231   
4                       Pharmacie Zanane صيدلية زنان  Beni Mellal  32.333186   

        lon          phone              website brand      type  
0 -6.350799           None                 

In [77]:
import requests
import pandas as pd
import time

# =========================
# 1. VILLES MAROC + BBOX
# =========================
cities = {
    "Casablanca": (33.48, -7.75, 33.75, -7.45),
    "Rabat": (33.90, -6.90, 34.05, -6.75),
    "Marrakech": (31.55, -8.10, 31.75, -7.85),
    "Fès": (34.00, -5.10, 34.10, -4.90),
    "Tanger": (35.70, -5.95, 35.85, -5.70),
    "Agadir": (30.35, -9.60, 30.50, -9.40),
    "Meknès": (33.85, -5.60, 34.00, -5.40),
    "Oujda": (34.65, -1.95, 34.80, -1.80),
    "Kenitra": (34.22, -6.70, 34.32, -6.55),
    "Tétouan": (35.55, -5.45, 35.65, -5.30),
    "Safi": (32.25, -9.25, 32.35, -9.10),
    "El Jadida": (33.20, -8.55, 33.30, -8.40),
    "Nador": (35.10, -2.95, 35.20, -2.80),
    "Beni Mellal": (32.30, -6.40, 32.40, -6.25),
    "Essaouira": (31.50, -9.80, 31.60, -9.65),
    "Ouarzazate": (30.90, -6.95, 31.05, -6.80),
    "Berkane": (34.90, -2.40, 35.00, -2.25),
    "Larache": (35.15, -6.15, 35.25, -6.00),
    "Settat": (33.00, -7.65, 33.10, -7.50)
}

# =========================
# 2. CONFIG
# =========================
url = "https://overpass-api.de/api/interpreter"
headers = {"User-Agent": "Mozilla/5.0"}

all_data = []

# =========================
# 3. LOOP VILLES
# =========================
for city, bbox in cities.items():
    print(f"\n🛒 Scraping supermarkets {city} ...")

    query = f"""
    [out:json][timeout:180];
    (
      node["shop"="supermarket"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
      way["shop"="supermarket"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});

      node["shop"="convenience"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
      way["shop"="convenience"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
    );
    out center;
    """

    data = None

    # =========================
    # 4. RETRY SYSTEM
    # =========================
    for i in range(3):
        try:
            r = requests.post(
                url,
                data={"data": query},
                headers=headers,
                timeout=240
            )

            print(f"{city} Status:", r.status_code)

            if r.status_code in [429, 504, 405]:
                print(f"⚠️ retry {city}")
                time.sleep(15 * (i + 1))
                continue

            if r.status_code != 200:
                print(f"❌ error {city}")
                time.sleep(10)
                continue

            if "json" not in r.headers.get("Content-Type", ""):
                print(f"❌ non JSON {city}")
                time.sleep(10)
                continue

            data = r.json()
            break

        except Exception as e:
            print(f"Retry error {city}:", e)
            time.sleep(10)

    if not data or not data.get("elements"):
        print(f"❌ No data {city}")
        continue

    # =========================
    # 5. PARSING
    # =========================
    for el in data["elements"]:
        tags = el.get("tags", {})

        name = tags.get("name")
        if not name:
            continue

        lat = el.get("lat", el.get("center", {}).get("lat"))
        lon = el.get("lon", el.get("center", {}).get("lon"))

        if not lat or not lon:
            continue

        all_data.append({
            "name": name,
            "city": city,
            "lat": lat,
            "lon": lon,
            "type": tags.get("shop"),  # supermarket ou convenience
            "brand": tags.get("brand"),
            "phone": tags.get("phone"),
            "website": tags.get("website"),
            "opening_hours": tags.get("opening_hours")
        })

    print(f"✅ {city} done | total shops: {len(all_data)}")

    time.sleep(12)

# =========================
# 6. DATAFRAME FINAL
# =========================
df = pd.DataFrame(all_data)

df = df.dropna(subset=["name"]).drop_duplicates()

print("\n📊 FINAL SUPERMARKET DATASET")
print(df.shape)
print(df.head())

# =========================
# 7. EXPORT CSV
# =========================
df.to_csv("morocco_supermarkets.csv", index=False, encoding="utf-8-sig")

print("\n✅ CSV saved: morocco_supermarkets.csv")


🛒 Scraping supermarkets Casablanca ...
Casablanca Status: 200
✅ Casablanca done | total shops: 150

🛒 Scraping supermarkets Rabat ...
Rabat Status: 504
⚠️ retry Rabat
Rabat Status: 200
✅ Rabat done | total shops: 234

🛒 Scraping supermarkets Marrakech ...
Marrakech Status: 429
⚠️ retry Marrakech
Marrakech Status: 200
✅ Marrakech done | total shops: 279

🛒 Scraping supermarkets Fès ...
Fès Status: 504
⚠️ retry Fès
Fès Status: 504
⚠️ retry Fès
Fès Status: 200
✅ Fès done | total shops: 508

🛒 Scraping supermarkets Tanger ...
Tanger Status: 200
✅ Tanger done | total shops: 556

🛒 Scraping supermarkets Agadir ...
Agadir Status: 200
✅ Agadir done | total shops: 578

🛒 Scraping supermarkets Meknès ...
Meknès Status: 504
⚠️ retry Meknès
Meknès Status: 504
⚠️ retry Meknès
Meknès Status: 200
✅ Meknès done | total shops: 614

🛒 Scraping supermarkets Oujda ...
Oujda Status: 504
⚠️ retry Oujda
Oujda Status: 504
⚠️ retry Oujda
Oujda Status: 200
✅ Oujda done | total shops: 622

🛒 Scraping supermarke

In [83]:
import requests
import pandas as pd
import time

# =========================
# 1. VILLES MAROC + BBOX
# =========================
cities = {
    "El Jadida": (33.20, -8.55, 33.30, -8.40),
    "Larache": (35.15, -6.15, 35.25, -6.00),
    "Settat": (33.00, -7.65, 33.10, -7.50)
}

# =========================
# 2. CONFIG
# =========================
url = "https://overpass-api.de/api/interpreter"
headers = {"User-Agent": "Mozilla/5.0"}

all_data = []

# =========================
# 3. LOOP VILLES
# =========================
for city, bbox in cities.items():
    print(f"\n🛒 Scraping supermarkets {city} ...")

    query = f"""
    [out:json][timeout:180];
    (
      node["shop"="supermarket"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
      way["shop"="supermarket"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});

      node["shop"="convenience"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
      way["shop"="convenience"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
    );
    out center;
    """

    data = None

    # =========================
    # 4. RETRY SYSTEM
    # =========================
    for i in range(3):
        try:
            r = requests.post(
                url,
                data={"data": query},
                headers=headers,
                timeout=240
            )

            print(f"{city} Status:", r.status_code)

            if r.status_code in [429, 504, 405]:
                print(f"⚠️ retry {city}")
                time.sleep(15 * (i + 1))
                continue

            if r.status_code != 200:
                print(f"❌ error {city}")
                time.sleep(10)
                continue

            if "json" not in r.headers.get("Content-Type", ""):
                print(f"❌ non JSON {city}")
                time.sleep(10)
                continue

            data = r.json()
            break

        except Exception as e:
            print(f"Retry error {city}:", e)
            time.sleep(10)

    if not data or not data.get("elements"):
        print(f"❌ No data {city}")
        continue

    # =========================
    # 5. PARSING
    # =========================
    for el in data["elements"]:
        tags = el.get("tags", {})

        name = tags.get("name")
        if not name:
            continue

        lat = el.get("lat", el.get("center", {}).get("lat"))
        lon = el.get("lon", el.get("center", {}).get("lon"))

        if not lat or not lon:
            continue

        all_data.append({
            "name": name,
            "city": city,
            "lat": lat,
            "lon": lon,
            "type": tags.get("shop"),  # supermarket ou convenience
            "brand": tags.get("brand"),
            "phone": tags.get("phone"),
            "website": tags.get("website"),
            "opening_hours": tags.get("opening_hours")
        })

    print(f"✅ {city} done | total shops: {len(all_data)}")

    time.sleep(12)

# =========================
# 6. DATAFRAME FINAL
# =========================
df = pd.DataFrame(all_data)

df = df.dropna(subset=["name"]).drop_duplicates()

print("\n📊 FINAL SUPERMARKET DATASET")
print(df.shape)
print(df.head())

# =========================
# 7. EXPORT CSV
# =========================
df.to_csv("morocco_supermarkets11.csv", index=False, encoding="utf-8-sig")

print("\n✅ CSV saved: morocco_supermarkets.csv")


🛒 Scraping supermarkets El Jadida ...
El Jadida Status: 200
✅ El Jadida done | total shops: 12

🛒 Scraping supermarkets Larache ...
Larache Status: 504
⚠️ retry Larache
Larache Status: 504
⚠️ retry Larache
Larache Status: 200
❌ No data Larache

🛒 Scraping supermarkets Settat ...
Settat Status: 200
❌ No data Settat

📊 FINAL SUPERMARKET DATASET
(12, 9)
               name       city        lat       lon         type  \
0             Acima  El Jadida  33.243459 -8.513328  supermarket   
1           Marjane  El Jadida  33.224187 -8.530470  supermarket   
2               Bim  El Jadida  33.234939 -8.493713  supermarket   
3             Acima  El Jadida  33.243268 -8.513465  supermarket   
4  Carrefour Market  El Jadida  33.239740 -8.496301  supermarket   

              brand phone website opening_hours  
0              None  None    None          None  
1           Marjane  None    None          None  
2               Bim  None    None          None  
3              None  None    None    